# 🔍 Resume Checker

## Learning Objectives
In this notebook, you will learn:
1. **PDF Resume Parsing** - Extract text from a PDF resume using PyPDFLoader
2. **Evaluation Prompt Design** - Create a structured prompt for multi-criteria resume analysis
3. **LangChain PromptTemplate** - Use input variables to build reusable prompt templates
4. **Streaming Analysis** - Generate and display the evaluation in real-time

## Prerequisites
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (the notebook uses `gpt-4o-mini` through `aied_helpers.get_openai_llm`)
- A candidate resume in PDF format at `content/Resume.pdf` (a sample ships with the project; replace it with yours)

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model through the course's LLM factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The model that evaluates the resume
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Resume Loading

Load the candidate's PDF resume and extract its text content.

In [ ]:
# ============================================================================
# RESUME LOADING: Extract Text from PDF
# ============================================================================
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../content/Resume.pdf")
documents = loader.load()

context = "\n\n".join(doc.page_content for doc in documents)

print(f"📄 Loaded {len(documents)} page(s) from resume")

---
## 📊 Part 2: Evaluation Prompt

Define a structured evaluation prompt that scores the resume across multiple criteria.

### Key Concepts:
- **PromptTemplate** allows reusable prompts with variable injection — every declared variable (`{context}`, `{question}`) must appear in the template text, or its value is silently dropped
- **Multi-criteria scoring** evaluates clarity, relevance, format, comprehensiveness, and keywords
- **Actionable output** includes strengths, weaknesses, and career path suggestions

In [ ]:
# ============================================================================
# EVALUATION PROMPT: Multi-Criteria Resume Analysis Template
# ============================================================================
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
    You are an advanced resume evaluation assistant. Analyze the provided
    resume in the context document and score it out of 100 based on the
    following criteria: clarity, relevance, format, comprehensiveness,
    and keywords.

    Your response should be structured as follows:

    1. **Score**: Provide the score out of 100.
    2. **Strengths**: List at least three strengths found in the resume.
    3. **Weaknesses**: List at least three weaknesses or areas for
       improvement in the resume.
    4. **Skills Mentioned**: Identify and list the skills that are
       explicitly mentioned in the resume.
    5. **Recommended Skills**: Suggest additional skills that could
       enhance the resume's effectiveness.
    6. **Next Career Path**: Suggest what should be the next career
       paths for this candidate.

    Please ensure your analysis is clear and concise, providing
    actionable insights for improvement.

    Request: {question}

    Resume:
    {context}
    """
)

print(f"✅ Evaluation prompt template configured with variables: {prompt.input_variables}")

---
## 🚀 Part 3: Run Resume Evaluation

Format the prompt with the resume context and stream the evaluation results.

In [ ]:
# ============================================================================
# EVALUATE: Format Prompt and Stream Results
# ============================================================================
question = "Please evaluate this resume."
formatted_prompt = prompt.format(context=context, question=question)

# Check that the resume text really made it into the prompt sent to the model
print(f"📏 Prompt: {len(formatted_prompt)} chars, resume included: {context.strip()[:50] in formatted_prompt}")

print("🔍 Evaluating resume...\n")

for m in llm.stream(formatted_prompt):
    print(m.content, end="", flush=True)

print("\n\n✅ Resume evaluation complete!")

---
## 📝 Summary

In this notebook, we learned:

### 1. PDF Resume Parsing
- **PyPDFLoader** extracts multi-page resume content for automated analysis

### 2. Structured Evaluation Prompts
- **PromptTemplate** with input variables creates reusable, parameterized prompts
- **Multi-criteria scoring** provides a comprehensive resume assessment

### 3. Actionable Feedback
- The evaluation covers **strengths, weaknesses, skills, and career paths**
- Output is structured for easy consumption and action

### Next Steps
- **`Resume_Scorer.ipynb`**: adds a job description and scores the same resume for ATS fit against it